# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [1]:
import requests
from bs4 import BeautifulSoup
import time
import numpy as np
import re
import pandas as pd

import requests
from bs4 import BeautifulSoup

#https://realpython.com/beautiful-soup-web-scraper-python/

def get_urls(page_url, urls=None, not_correct_urls=None):
    headers = { #We need this to be able to have access pretty much to get the data.
        "User-Agent": (
            "Chrome/124.0.0.0 "
            "Edg/150.0.4078.48"
        )
    }

    content = requests.get(page_url, headers=headers)
    soup = BeautifulSoup(content.content, "html.parser") #uses html parser for the response because it is html and create the BeautifulSoup object with content as content as input.

    if not urls: #I create a new set if it does not exist
        urls = set()
    if not not_correct_urls: #I create a new set if it does not exist
        not_correct_urls = set()

    for a in soup.find_all("a", href=True): #I get all the a tags
        href = a["href"]
        href = re.sub(r"^(?:\.\./)+", "/", href) #The paths are like ../../../ so I am replacing them with a / so I can add the actual url
        href = "https://www.fixr.com" + href
        # Fixr cost pages always contain /costs/ , so we are looking for these

        if "/costs/" in href: #If this exists than we want to add it unless it already exists. i know that it is a set and duplicates cannot exist but I want to be careful
            if href not in urls:
                urls.add(href)
        else: #if not, add the href to the not correct urls set.
            if href not in not_correct_urls:
                not_correct_urls.add(href)
    return urls, not_correct_urls

In [2]:
def scrape_data(url):
    headers = {
        "User-Agent": (
            "Chrome/124.0.0.0 "
            "Edg/150.0.4078.48"
        )
    }
    #doing same thing
    content = requests.get(url, headers=headers).text
    soup = BeautifulSoup(content, "html.parser")

    data = { #dict containing the description and url
        "url": url,
        "description": None,
    }

    # Extract average cost
    for p in soup.find_all("p"): #the description is in the p tag. In the description, it has text containing the national average cost
        #only if the p has "average" in the text
        clean_text = p.get_text(strip=True).lower()
        if "average" in clean_text:
            #they could be using average but talking about something else, so we check that the dollar sign is in the text.
            if "$" in clean_text:
                data["description"] = clean_text #we add it to the dict.
                break #break out of loop

    return data


In [3]:
url_set, not_in_set = get_urls("https://www.fixr.com/costs/")

for url in not_in_set.copy(): #I got a error when I didn't use a copy of the not_in_set because I am updating it during the loop. so I am using copy.
    new_set, new_not_in_set = get_urls(url, url_set, not_in_set) # I get urls from the not_in_set
    for u in new_set:
        if u not in url_set:
            url_set.add(u) #I add to url set if it is in new set. (this will inlcude that past urls already in url_set, so we use this condition just in case even thought it is a set)

    for u in new_not_in_set:
        not_in_set.add(u) #I add to not_in_set set if it is in new set. (this will inlcude that past urls already in not_in_set, so we use this condition just in case even thought it is a set)

urls = dict() #we initialize an empty dict
for url in url_set: #I am getting the last part of the https:____/__/name
   name = url.split("/")[-1]
   urls[name] = url #I add this to my dict with the name as key and url as value

dataset = {name: scrape_data(url) for name, url in urls.items()} #for each url, I am scraping the data from it and soring it in this dict as my dataset..

In [6]:
rows = []
for name, data in dataset.items(): #name is the key and data is the value.
    row = { #I am looping through this and setting up a dict for creating the dataframe
        "item": name,
        "url": data["url"],
        "description": data["description"]
    }
    rows.append(row) #I append to the rows list

df = pd.DataFrame(rows) #I create the dataframe from the rows
df["description"] = df["description"].str.strip().str.lower() #making sure the description is cleaned.

df["upgrades"] = np.zeros(df["description"].shape) #I am adding categories and filling them with 0 to start
df["repairs"] = np.zeros(df["description"].shape)
df["bidirectional"] = np.zeros(df["description"].shape)

df["upgrades"] = df["upgrades"].astype("category")
df["repairs"] = df["repairs"].astype("category")
df["bidirectional"] = df["bidirectional"].astype("category")

In [7]:
avg_cost_list = []
na_idx = []

for i, row in enumerate(df["description"]):
    if row:
        row = row.replace(",", "")
        # https://www.rexegg.com/regex-quantifiers.php
        avg_val = re.findall(r"average.*?\$([\d,]+)", row)
        avg_val = np.array(avg_val)
        if avg_val.shape != (1,):
            na_idx.append(i)
        else:
            avg_cost_list.append(float(avg_val[0]))
    else:
        na_idx.append(i)

df = df.drop(index=na_idx).reset_index(drop=True)
df["avg_cost"] = avg_cost_list


In [9]:
df

,item,url,description,upgrades,repairs,bidirectional,avg_cost
0,hot-tub-installation,https://www.fixr.com/costs/hot-tub-installation,"hot tub installation costs range from$4,000 to...",0.0,0.0,0.0,6900.0
1,kitchen-island-installation,https://www.fixr.com/costs/kitchen-island-inst...,"kitchen island cost ranges from$1,500 to $8,00...",0.0,0.0,0.0,5000.0
2,shower-door-installation,https://www.fixr.com/costs/shower-door-install...,"walk-in shower installation costs$5,000 to $11...",0.0,0.0,0.0,8000.0
3,mosquito-misting-system,https://www.fixr.com/costs/mosquito-misting-sy...,"mosquito misting systems cost between$1,800 an...",0.0,0.0,0.0,2500.0
4,saltwater-pool,https://www.fixr.com/costs/saltwater-pool,"saltwater pool costs range from$30,000 to $70,...",0.0,0.0,0.0,50000.0
...,...,...,...,...,...,...,...
790,install-roof-shingles,https://www.fixr.com/costs/install-roof-shingles,"since 2022, shingleroof replacement costs are ...",0.0,0.0,0.0,14500.0
791,evaporative-cooler-repair,https://www.fixr.com/costs/evaporative-cooler-...,"swamp cooler repair costs$115 to $350, with an...",0.0,0.0,0.0,250.0
792,digital-phone-system,https://www.fixr.com/costs/digital-phone-system,"telephone system costs range from$50 to $265, ...",0.0,0.0,0.0,100.0
793,building-permits,https://www.fixr.com/costs/building-permits,"building permit costs range from$450 to $700, ...",0.0,0.0,0.0,550.0


In [8]:
# df.loc[:, "repair_item"] = df["repair_item"].str.replace(r"(?:-|#)", " ", regex=True) #I split based on the //. I want the last one because it is name of the page. I replace any of the special characters with a space
#
# index = df[df["repair_item"].str.contains("roof replacement ")].index
# df = df.drop(index=index).reset_index(drop=True)
#
# index = df[df["repair_item"].str.contains("sqft")].index
# df = df.drop(index=index).reset_index(drop=True)
#
# mask = df["repair_item"].astype(str).str.contains(r"\b(?:repair|refin|maint|replace|install|remov|exterm|clean|recov|remod|add)\w*", regex=True)
# df.loc[mask, "repair_item"] = df.loc[mask, "repair_item"].str.replace(r"\b(?:repair|refin|maint|replace|install|remov|exterm|clean|recov|remod|add)\w*", "", regex=True)
#
# index = df[df["repair_item"].astype(str).str.contains("treatment-")].index
# df = df.drop(index=index).reset_index(drop=True)
#
# index = df[df["repair_item"].astype(str).str.contains("solar")]["repair_item"].index
# df.loc[index[-1], "repair_item"] = "solar panel"
# df = df.drop(index=index[:-1]).reset_index(drop=True)
#
# index = df[df["repair_item"].astype(str).str.contains("carpet")]["repair_item"].index
# df.loc[index[-1], "repair_item"] = "carpet"
# df = df.drop(index=index[:-1]).reset_index(drop=True)
#
# index = df[(df["avg_cost"] > 50000) & ~(df["repair_item"].str.contains(r"pool"))].index
# df = df.drop(index=index).reset_index(drop=True)

In [ ]:
# cleaned_df = df[["repair_item", "avg_cost"]]
# cleaned_df = cleaned_df.drop_duplicates().reset_index(drop=True)

In [ ]:
# cleaned_df.to_csv("cleaned_repairs.csv", index=False)